# Colab 2-Stage 파이프라인 튜토리얼

**목적**: Google Colab에서 YOLO26n 기반 약물 탐지(Stage 1) + crop 분류(Stage 2) 파이프라인을 실행하고, Validation E2E 평가와 Kaggle 제출 CSV 생성까지 확인합니다.

로컬 와 같은 교육 흐름을 유지하되, repo clone / Drive mount / data unzip만 Colab 기준으로 바꾼 버전입니다.

> Kaggle 제출은 팀 규칙에 따라 별도로 진행합니다. 이 노트북은 제출 파일 생성까지만 다룹니다.

## 1. 유틸 함수 지도

튜토리얼에서 자주 쓰는 시각화/리포트 유틸은 `src.utils.visualize`, `src.utils.report`에 모여 있습니다.

| 함수 | 용도 |
|---|---|
| `plot_training_curves` | Stage 1 loss / mAP 학습 곡선 확인 |
| `plot_s1_gt_vs_pred` | Stage 1 GT bbox와 예측 bbox 비교 |
| `plot_crop_showcase`, `plot_crop_grid` | crop 품질 및 샘플 확인 |
| `plot_pipeline_overlay`, `plot_pipeline_strip` | bbox + Stage 2 class 통합 결과 시각화 |
| `print_timings` | 실험 폴더의 `timings.json` 요약 |

## 2. 파이프라인 개요

```text
원본 이미지
    ↓
[Stage 1: YOLO26n 탐지] 이미지 → bbox 리스트
    ↓
[Crop] bbox 기준 개별 알약 이미지 생성
    ↓
[Stage 2: 분류] crop → 약품명 예측
    ↓
[E2E 평가 / Kaggle 제출] bbox + class + score 병합
```

| 단계 | 입력 | 출력 |
|---|---|---|
| Stage 1 추론 | val/test 원본 이미지 | `*_predictions.json` |
| Crop 생성 | Stage 1 predictions + 원본 이미지 | `crops_manifest.json`, crop 이미지 |
| Stage 2 추론 | inference crop | `stage2_predictions.json` |
| E2E 평가 | GT label/image + crop manifest + Stage 2 predictions | mAP |
| 제출 생성 | test crop manifest + Stage 2 predictions | `submission.csv` |

## 3. Colab Bootstrap

아래 셀은 Drive mount, repo clone, dependency install, Drive zip 데이터 압축 해제를 수행합니다.

필요하면 사용자 설정만 수정하세요. 특히 `DRIVE_DATA_ZIP`, `FORCE_RECLONE`, `FORCE_UNZIP_DATA`를 확인합니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import zipfile
import subprocess
import sys
try:
    import koreanize_matplotlib  # noqa: F401
except ImportError:
    subprocess.run(["pip", "install", "-q", "koreanize-matplotlib"])
    import koreanize_matplotlib  # noqa: F401, E402


# =========================
# 0. 사용자 설정
# =========================

REPO_URL = "https://github.com/Codeit-Part2-2Team/codeit-part2-2team-project"
PROJECT_ROOT = Path("/content/codeit-part2-2team-project")

DRIVE_ROOT = Path("/content/drive/MyDrive/GDriveAutoShare_Downloads")
DRIVE_DATA_ZIP = DRIVE_ROOT / "C:\\Users\\user\\Desktop\\processed_v6.zip"

LOCAL_DATA_ROOT = Path("/content/data")
EXPECTED_DATA_YAML = LOCAL_DATA_ROOT / "processed/dataset.yaml"
EXPECTED_CROP_ROOT = LOCAL_DATA_ROOT / "processed"

RUN_BOOTSTRAP = True       # False: Drive mount / clone / install / unzip 전부 스킵
INSTALL_REQUIREMENTS = True
INSTALL_EDITABLE = True
FORCE_RECLONE = False
FORCE_UNZIP_DATA = False

if RUN_BOOTSTRAP:
    # =========================
    # 1. Drive mount
    # =========================

    drive.mount("/content/drive")

    # =========================
    # 2. Repo 준비
    # =========================

    if FORCE_RECLONE and PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)

    if not PROJECT_ROOT.exists():
        print("[repo] cloning...")
        subprocess.run(
            ["git", "clone", REPO_URL, str(PROJECT_ROOT)],
            check=True,
        )
    else:
        print("[repo] already exists:", PROJECT_ROOT)

    os.chdir(PROJECT_ROOT)
    print("[cwd]", Path.cwd())

    # =========================
    # 3. Python dependencies
    # =========================

    if INSTALL_REQUIREMENTS:
        print("[install] requirements.txt")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-r", "requirements.txt"],
            check=True,
        )

    if INSTALL_EDITABLE:
        print("[install] editable")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-e", "."],
            check=True,
        )

    # =========================
    # 4. Data 준비
    # =========================

    print("[data] zip:", DRIVE_DATA_ZIP)
    if not DRIVE_DATA_ZIP.exists():
        raise FileNotFoundError(f"Drive data zip not found: {DRIVE_DATA_ZIP}")

    need_unzip = FORCE_UNZIP_DATA or not EXPECTED_DATA_YAML.exists()

    if need_unzip:
        LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
        local_zip = Path("/content/processed_dataset.zip")

        print("[data] copy zip to local:", local_zip)
        shutil.copy2(DRIVE_DATA_ZIP, local_zip)

        print("[data] unzip to:", LOCAL_DATA_ROOT)
        with zipfile.ZipFile(local_zip, "r") as z:
            z.extractall(LOCAL_DATA_ROOT)

        # Optional nested zip: processed_v6.zip 안에 processed.zip이 있는 경우
        nested_zip = LOCAL_DATA_ROOT / "processed.zip"
        if nested_zip.exists():
            print("[data] nested zip found, unzip:", nested_zip)
            with zipfile.ZipFile(nested_zip, "r") as z:
                z.extractall(LOCAL_DATA_ROOT)
    else:
        print("[data] already prepared:", LOCAL_DATA_ROOT)

    # Repo-relative data path symlink
    repo_data = PROJECT_ROOT / "data"
    if not repo_data.exists():
        print("[data] symlink repo data ->", LOCAL_DATA_ROOT)
        repo_data.symlink_to(LOCAL_DATA_ROOT, target_is_directory=True)
    else:
        print("[data] repo data already exists:", repo_data)

else:
    print("[bootstrap] 스킵 — 이미 환경이 준비되어 있다고 가정합니다.")
    os.chdir(PROJECT_ROOT)
    print("[cwd]", Path.cwd())

# =========================
# 5. 경로 검증
# =========================

print("\n=== PATH CHECK ===")
print("PROJECT_ROOT:", PROJECT_ROOT, PROJECT_ROOT.exists())
print("requirements:", (PROJECT_ROOT / "requirements.txt").exists())
print("scripts:", (PROJECT_ROOT / "scripts").exists())
print("src:", (PROJECT_ROOT / "src").exists())

print("LOCAL_DATA_ROOT:", LOCAL_DATA_ROOT, LOCAL_DATA_ROOT.exists())
print("EXPECTED_DATA_YAML:", EXPECTED_DATA_YAML, EXPECTED_DATA_YAML.exists())
print("EXPECTED_CROP_ROOT:", EXPECTED_CROP_ROOT, EXPECTED_CROP_ROOT.exists())
print("crop train:", (EXPECTED_CROP_ROOT / "crops/train").exists() or (EXPECTED_CROP_ROOT / "train").exists())
print("crop val:", (EXPECTED_CROP_ROOT / "crops/val").exists() or (EXPECTED_CROP_ROOT / "val").exists())
print("crop test:", (EXPECTED_CROP_ROOT / "crops/test").exists() or (EXPECTED_CROP_ROOT / "test").exists())


## 4. 경로 및 실행 옵션

Colab에서는 repo와 데이터가 분리되어 있으므로 절대경로를 명시합니다. repo 내부 `data` symlink도 bootstrap에서 맞춰두지만, 아래 변수들은 항상 Colab 절대경로를 기준으로 사용합니다.

In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys
import pandas as pd
import torch
import yaml

PROJECT_ROOT = Path("/content/codeit-part2-2team-project")
os.chdir(PROJECT_ROOT)

EXP_NAME = "exp_20260420_baseline_yolo26n"
EXP_DIR = PROJECT_ROOT / "experiments" / EXP_NAME

DATA_ROOT = Path("/content/data/processed")
DATA_YAML = DATA_ROOT / "dataset.yaml"
VAL_IMG_DIR = DATA_ROOT / "images/val"
VAL_LABEL_DIR = DATA_ROOT / "labels/val"
TEST_IMG_DIR = Path("/content/data/test_images")
if not TEST_IMG_DIR.exists():
    TEST_IMG_DIR = DATA_ROOT / "images/test"

# processed_v6.zip 구조에 따라 crop root가 data/processed/crops 또는 data/processed 일 수 있다.
if (DATA_ROOT / "crops/train/crops_manifest.json").exists():
    GT_CROP_ROOT = DATA_ROOT / "crops"
elif (DATA_ROOT / "train/crops_manifest.json").exists():
    GT_CROP_ROOT = DATA_ROOT
else:
    GT_CROP_ROOT = DATA_ROOT / "crops"

S1_CONFIG = EXP_DIR / "s1_config.yaml"
S2_CONFIG = EXP_DIR / "s2_config.yaml"
if not S2_CONFIG.exists():
    S2_CONFIG = PROJECT_ROOT / "experiments/stage2_classifier/config.yaml"

BEST_PT_S1 = EXP_DIR / "weights/best.pt"
BEST_PT_S2 = EXP_DIR / "stage2/weights/best.pt"
if not BEST_PT_S2.exists():
    BEST_PT_S2 = PROJECT_ROOT / "experiments/stage2_classifier/weights/best.pt"

VAL_S1_PRED = EXP_DIR / "val_predictions.json"
VAL_CROP_DIR = EXP_DIR / "stage1_crops"
VAL_S2_PRED = EXP_DIR / "stage2_predictions.json"

TEST_S1_PRED = EXP_DIR / "test_s1_predictions.json"
TEST_CROP_DIR = EXP_DIR / "test_s1_crops"
TEST_S2_PRED = EXP_DIR / "test_s2_predictions.json"
SUBMISSION_CSV = PROJECT_ROOT / "submissions/submission_colab.csv"

KAGGLE_CLASS_MAP = PROJECT_ROOT / "data/processed/kaggle_class_map.json"
UNKNOWN_CLASS_MAP = PROJECT_ROOT / "data/processed/kaggle_unknown_class_map.json"

# Run All 안전 기본값: 학습/튜닝은 꺼둔다.
RUN_TRAIN_S1 = False
RUN_TRAIN_S2 = False
RUN_VAL_PIPELINE = True
RUN_TEST_SUBMISSION = False
RUN_STAGE2_TUNING = False

print("cwd:", Path.cwd())
for name, p in {
    "DATA_YAML": DATA_YAML,
    "VAL_IMG_DIR": VAL_IMG_DIR,
    "VAL_LABEL_DIR": VAL_LABEL_DIR,
    "GT_CROP_ROOT": GT_CROP_ROOT,
    "S1_CONFIG": S1_CONFIG,
    "S2_CONFIG": S2_CONFIG,
    "BEST_PT_S1": BEST_PT_S1,
    "BEST_PT_S2": BEST_PT_S2,
    "KAGGLE_CLASS_MAP": KAGGLE_CLASS_MAP,
    "UNKNOWN_CLASS_MAP": UNKNOWN_CLASS_MAP,
}.items():
    print(f"{name:18s}", "OK" if Path(p).exists() else "MISSING", p)

print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


def run_step(name, cmd, check=True, capture_output=False, **kwargs):
    print(f"\n[{name}]")
    print(" ".join(map(str, cmd)))
    result = subprocess.run(
        cmd,
        check=check,
        text=True,
        capture_output=capture_output,
        **kwargs,
    )
    if capture_output:
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print('[stderr]', result.stderr[:1000])
    return result


### 설정 파일 확인

Stage 1 / Stage 2의 핵심 설정을 출력합니다. seed, epoch, batch, image size가 의도와 맞는지 먼저 확인합니다.

In [ ]:
import yaml

if S1_CONFIG.exists():
    with open(S1_CONFIG, encoding='utf-8') as f:
        s1_cfg = yaml.safe_load(f)
    print('Stage 1 주요 설정')
    print('-' * 50)
    print('model       :', s1_cfg.get('model', {}).get('name'))
    print('imgsz       :', s1_cfg.get('data', {}).get('imgsz'))
    print('epochs      :', s1_cfg.get('train', {}).get('epochs'))
    print('batch       :', s1_cfg.get('train', {}).get('batch'))
    print('seed        :', s1_cfg.get('seed'))
else:
    print('S1_CONFIG 없음:', S1_CONFIG)

if S2_CONFIG.exists():
    with open(S2_CONFIG, encoding='utf-8') as f:
        s2_cfg = yaml.safe_load(f)
    print()
    print('Stage 2 주요 설정')
    print('-' * 50)
    print('model       :', s2_cfg.get('model', {}).get('name'))
    print('epochs      :', s2_cfg.get('train', {}).get('epochs'))
    print('batch       :', s2_cfg.get('train', {}).get('batch'))
    print('lr0         :', s2_cfg.get('train', {}).get('lr0'))
    print('seed        :', s2_cfg.get('seed'))
else:
    print('S2_CONFIG 없음:', S2_CONFIG)

### 데이터셋 확인

YOLO 형식 데이터셋 구조와 GT crop manifest 존재 여부를 확인합니다.

In [ ]:
if not DATA_YAML.exists():
    raise FileNotFoundError(f'DATA_YAML not found: {DATA_YAML}')

with open(DATA_YAML, encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

DATA_YAML_DIR = DATA_YAML.parent
print('데이터셋 구성')
print('-' * 70)
for split in ['train', 'val', 'test']:
    value = data_cfg.get(split)
    split_path = (DATA_YAML_DIR / value).resolve() if value else None
    count = len(list(split_path.glob('*.jpg'))) + len(list(split_path.glob('*.png'))) if split_path and split_path.exists() else -1
    print(f'{split:6s}: {count:5d} images  ->  {split_path}')

for split in ['train', 'val', 'test']:
    manifest_path = GT_CROP_ROOT / split / 'crops_manifest.json'
    print(f'GT crop manifest {split:5s}:', 'OK' if manifest_path.exists() else 'MISSING', manifest_path)

## 5. Stage 1: 약물 탐지 모델

### Step 1-1: 모델 학습 및 Validation 추론/Crop 생성

기본적으로 이미 학습된 Stage 1 weight를 사용합니다. Colab에서 새로 학습하려면 `RUN_TRAIN_S1=True`로 바꾼 뒤 실행합니다. 이어서 validation 이미지에 대해 Stage 1 추론과 crop 생성을 수행합니다.

In [ ]:
if RUN_TRAIN_S1:
    run_step("stage1_train", [
        sys.executable, "scripts/train.py",
        "--config", str(S1_CONFIG),
        "--data", str(DATA_YAML),
    ], check=True)
else:
    print("Stage 1 학습 스킵")

if RUN_VAL_PIPELINE:
    if not BEST_PT_S1.exists():
        raise FileNotFoundError(f"Stage 1 weight not found: {BEST_PT_S1}")
    run_step("stage1_predict", [
        sys.executable, "scripts/predict.py",
        "--config", str(S1_CONFIG),
        "--weights", str(BEST_PT_S1),
        "--source", str(VAL_IMG_DIR),
        "--output", str(VAL_S1_PRED),
    ], check=True)
    run_step("crop", [
        sys.executable, "scripts/pipeline/crop.py",
        "--predictions", str(VAL_S1_PRED),
        "--source", str(VAL_IMG_DIR),
        "--output", str(VAL_CROP_DIR),
    ], check=True)
    print("val crop:", VAL_CROP_DIR)


### Step 1-2: 모델 검증

학습된 Stage 1 모델을 validation split에서 평가합니다. `Precision`, `Recall`, `mAP50`, `mAP50-95`를 확인합니다.

In [ ]:
if BEST_PT_S1.exists():
    validate_cmd = [
        sys.executable, 'scripts/validate.py',
        '--config', str(S1_CONFIG),
        '--data', str(DATA_YAML),
        '--weights', str(BEST_PT_S1),
    ]
    run_step('stage1_validate', validate_cmd, capture_output=True)
else:
    print('Stage 1 weight 없음:', BEST_PT_S1)

### Step 1-3: 학습 곡선 시각화

가 있으면 Stage 1 loss와 mAP 추이를 확인합니다.

In [ ]:
from src.utils.visualize import plot_training_curves

RESULTS_CSV = EXP_DIR / 'results.csv'
if RESULTS_CSV.exists():
    plot_training_curves(RESULTS_CSV, save_dir=EXP_DIR)
else:
    print('results.csv 없음:', RESULTS_CSV)

### Step 1-4: GT vs Prediction 시각화

Stage 1 추론 결과가 준비되면 GT bbox와 예측 bbox를 비교합니다.

In [ ]:
from src.utils.visualize import plot_s1_gt_vs_pred

if VAL_S1_PRED.exists():
    with open(VAL_S1_PRED, encoding='utf-8') as f:
        predictions = json.load(f)

if 'predictions' not in globals():
    print('predictions 없음 — Stage 1 validation 추론 셀을 먼저 실행하세요.')
elif not VAL_IMG_DIR.exists() or not VAL_LABEL_DIR.exists():
    print('val image/label 경로를 확인하세요:', VAL_IMG_DIR, VAL_LABEL_DIR)
else:
    plot_s1_gt_vs_pred(predictions, VAL_IMG_DIR, VAL_LABEL_DIR)

### Step 1-5: Manifest 역할 정리

| manifest 종류 | 생성 모드 | 주요 필드 | 용도 |
|---|---|---|---|
| GT crop manifest | GT label 기반 crop 또는 준비된 crop manifest | `crop_path`, `class_name`, 선택적으로 `image_id`, `bbox` | Stage 2 학습/분류 평가 |
| ImageFolder manifest | `crop.py --imagefolder ...` | `crop_path`, `class_name` | 이미 잘린 crop을 Stage 2 학습 인터페이스에 연결 |
| Inference crop manifest | `crop.py --predictions ... --source ...` | `crop_id`, `image_id`, `bbox`, `score` | 복원 시각화, E2E 평가, Kaggle 제출 |

Kaggle 제출에는 반드시 test 원본 이미지에서 Stage 1 예측으로 만든 inference crop manifest를 사용합니다.

### Step 1-6: Crop 품질 확인

탐지한 알약들이 제대로 crop되었는지 시각적으로 확인합니다.

In [ ]:
from src.utils.visualize import plot_crop_showcase, plot_crop_grid

manifest_path = VAL_CROP_DIR / 'crops_manifest.json'
if manifest_path.exists():
    with open(manifest_path, encoding='utf-8') as f:
        manifest = json.load(f)
    plot_crop_showcase(manifest, VAL_CROP_DIR, VAL_IMG_DIR)
else:
    print('manifest 없음 — Stage 1 crop 생성 셀을 먼저 실행하세요:', manifest_path)

if GT_CROP_ROOT.exists():
    print('GT crop sample')
    plot_crop_grid(GT_CROP_ROOT / 'train', n=16)

## 6. Stage 2: 약물 분류 모델

기본적으로 이미 학습된 Stage 2 weight를 사용합니다. 새로 학습하려면 `RUN_TRAIN_S2=True`로 바꿉니다.

### Step 2-0: GT Crop 확인

Stage 2 학습은 GT crop train/val을 사용합니다. Colab 데이터 zip에 이미 포함되어 있으면 생성하지 않고 그대로 사용합니다.

In [ ]:
train_manifest = GT_CROP_ROOT / 'train/crops_manifest.json'
val_manifest = GT_CROP_ROOT / 'val/crops_manifest.json'

if train_manifest.exists() and val_manifest.exists():
    with open(train_manifest, encoding='utf-8') as f:
        gt_train_manifest = json.load(f)
    with open(val_manifest, encoding='utf-8') as f:
        gt_val_manifest = json.load(f)
    print('GT crop 이미 존재')
    print('Train crop:', len(gt_train_manifest))
    print('Val crop  :', len(gt_val_manifest))
else:
    print('GT crop manifest 없음 — imagefolder 기반으로 생성합니다.')
    print('ImageFolder root:', GT_CROP_ROOT)
    run_step('gt_crop_manifest', [
        sys.executable, 'scripts/pipeline/crop.py',
        '--imagefolder', str(GT_CROP_ROOT),
        '--splits', 'train', 'val',
    ], check=True)
    with open(train_manifest, encoding='utf-8') as f:
        gt_train_manifest = json.load(f)
    with open(val_manifest, encoding='utf-8') as f:
        gt_val_manifest = json.load(f)
    print('Train crop:', len(gt_train_manifest))
    print('Val crop  :', len(gt_val_manifest))


### Step 2-1: 분류 모델 학습 및 Validation 추론

GT crop으로 Stage 2 분류기를 학습하거나, 기존 checkpoint를 사용해 validation inference crop을 분류합니다.


In [ ]:
if RUN_TRAIN_S2:
    run_step("stage2_train", [
        sys.executable, "scripts/pipeline/stage2_train.py",
        "--config", str(S2_CONFIG),
        "--data", str(GT_CROP_ROOT),
    ], check=True)
else:
    print("Stage 2 학습 스킵")

if RUN_VAL_PIPELINE:
    if not BEST_PT_S2.exists():
        raise FileNotFoundError(f"Stage 2 weight not found: {BEST_PT_S2}")
    run_step("stage2_predict", [
        sys.executable, "scripts/pipeline/stage2_predict.py",
        "--config", str(S2_CONFIG),
        "--weights", str(BEST_PT_S2),
        "--source", str(VAL_CROP_DIR),
        "--output", str(VAL_S2_PRED),
    ], check=True)
    print("val stage2 predictions:", VAL_S2_PRED)


### Step 2-2: Stage 2 체크포인트 지표 확인

저장된 checkpoint의 epoch, Top-1, Top-5, 학습 클래스 수를 확인합니다.

In [ ]:
if not BEST_PT_S2.exists():
    print('Stage 2 weight 없음:', BEST_PT_S2)
else:
    ckpt = torch.load(BEST_PT_S2, map_location='cpu')
    metrics = ckpt.get('metrics', {})
    class_names = ckpt.get('class_names', [])
    print('Stage 2 best epoch:', ckpt.get('epoch', -1) + 1)
    print('Top-1 Acc         :', metrics.get('top1_acc', ckpt.get('top1_acc', 'N/A')))
    print('Top-5 Acc         :', metrics.get('top5_acc', ckpt.get('top5_acc', 'N/A')))
    print('학습 클래스 수    :', len(class_names))

## 7. Validation E2E 평가

Kaggle 기준 class map과 Stage 2 alias map을 같이 적용합니다.

In [ ]:
if RUN_VAL_PIPELINE:
    cmd = [
        sys.executable, "scripts/pipeline/evaluate_pipeline.py",
        "--gt-labels", str(VAL_LABEL_DIR),
        "--gt-images", str(VAL_IMG_DIR),
        "--s1-crops", str(VAL_CROP_DIR / "crops_manifest.json"),
        "--s2-preds", str(VAL_S2_PRED),
        "--kaggle-classes", str(KAGGLE_CLASS_MAP),
    ]
    if UNKNOWN_CLASS_MAP.exists():
        cmd += ["--unknown-class-map", str(UNKNOWN_CLASS_MAP)]
    run_step("evaluate_pipeline", cmd, capture_output=True)

## 8. Kaggle Test 제출 CSV 생성

실제 제출 파일을 만들 때만 `RUN_TEST_SUBMISSION=True`로 바꿉니다. 이 셀은 test 이미지에 대해 Stage 1 추론, crop 생성, Stage 2 추론, submission CSV 생성을 수행합니다.

In [ ]:
if RUN_TEST_SUBMISSION:
    if not TEST_IMG_DIR.exists():
        raise FileNotFoundError(f"test image dir not found: {TEST_IMG_DIR}")
    run_step("stage1_predict", [
        sys.executable, "scripts/predict.py",
        "--config", str(S1_CONFIG),
        "--weights", str(BEST_PT_S1),
        "--source", str(TEST_IMG_DIR),
        "--output", str(TEST_S1_PRED),
    ], check=True)
    run_step("crop", [
        sys.executable, "scripts/pipeline/crop.py",
        "--predictions", str(TEST_S1_PRED),
        "--source", str(TEST_IMG_DIR),
        "--output", str(TEST_CROP_DIR),
    ], check=True)
    run_step("stage2_predict", [
        sys.executable, "scripts/pipeline/stage2_predict.py",
        "--config", str(S2_CONFIG),
        "--weights", str(BEST_PT_S2),
        "--source", str(TEST_CROP_DIR),
        "--output", str(TEST_S2_PRED),
    ], check=True)
    SUBMISSION_CSV.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, "scripts/make_submission.py",
        "--manifest", str(TEST_CROP_DIR / "crops_manifest.json"),
        "--s2-preds", str(TEST_S2_PRED),
        "--class-map", str(KAGGLE_CLASS_MAP),
        "--output", str(SUBMISSION_CSV),
    ]
    if UNKNOWN_CLASS_MAP.exists():
        cmd += ["--unknown-class-map", str(UNKNOWN_CLASS_MAP)]
    run_step("make_submission", cmd)
    print("submission:", SUBMISSION_CSV)
    display(pd.read_csv(SUBMISSION_CSV).head())
else:
    print("test submission 스킵: RUN_TEST_SUBMISSION=False")


## 9. 예측 시간 요약 및 결과 시각화

Colab에서 실행한 주요 단계의 소요 시간을 확인하고, Stage 1 bbox + Stage 2 class prediction을 원본 이미지에 overlay해 샘플 결과를 확인합니다.

In [ ]:
from src.utils.report import print_timings

print_timings(EXP_DIR)

In [ ]:
print("통합 overlay 시각화는 아래 10번 섹션에서 src.utils.visualize.plot_pipeline_overlay로 확인합니다.")

## 10. 2-Stage 통합 결과 구성

Stage 1 inference manifest와 Stage 2 predictions를 `crop_id`로 다시 묶어 이미지 단위 결과를 만듭니다. 이 구조는 overlay, strip 시각화와 디버깅에 사용합니다.

In [ ]:
manifest_path = VAL_CROP_DIR / 'crops_manifest.json'

if not manifest_path.exists():
    print('manifest 없음:', manifest_path)
elif not VAL_S2_PRED.exists():
    print('Stage 2 prediction 없음:', VAL_S2_PRED)
else:
    with open(manifest_path, encoding='utf-8') as f:
        manifest = json.load(f)
    with open(VAL_S2_PRED, encoding='utf-8') as f:
        s2_results = json.load(f)

    s2_by_crop = {r['crop_id']: r for r in s2_results}
    pipeline_by_img = {}
    missing_s2 = 0
    for item in manifest:
        s2 = s2_by_crop.get(item['crop_id'])
        if s2 is None:
            missing_s2 += 1
            s2 = {}
        pipeline_by_img.setdefault(item['image_id'], []).append({
            'bbox': item['bbox'],
            'det_score': item.get('score', 0.0),
            'crop_id': item['crop_id'],
            'class_name': s2.get('class_name', '?'),
            'class_score': s2.get('score', 0.0),
        })

    multi_count = sum(1 for rows in pipeline_by_img.values() if len(rows) >= 2)
    print('통합 결과 준비 완료:', len(pipeline_by_img), '장 / crop', len(manifest), '개')
    print('multi 탐지 이미지:', multi_count, '장')
    print('Stage 2 매칭 누락 crop:', missing_s2, '개')
    print('image_id 예시:', list(pipeline_by_img)[:3])

### 결과 시각화: 통합 Overlay

약물 위치에 Stage 2 약품명과 class score를 직접 표시합니다.

In [ ]:
import src.utils.visualize as viz
importlib.reload(viz)

if 'pipeline_by_img' not in globals():
    print('pipeline_by_img 없음 — 바로 위 통합 결과 구성 셀을 먼저 실행하세요.')
else:
    viz.plot_pipeline_overlay(
        pipeline_by_img,
        VAL_IMG_DIR,
        save_dir=EXP_DIR,
    )

## 11. 결과 해석 및 다음 단계

| 메트릭 | 의미 | 목표 방향 |
|---|---|---|
| S1 Precision / Recall | 탐지 품질 | 둘 다 높을수록 좋음 |
| S1 mAP50-95 | Stage 1 bbox 품질 | 엄격한 IoU에서 유지 필요 |
| S2 Top-1 / Top-5 | crop 분류 품질 | Stage 2 튜닝의 직접 objective |
| E2E `mAP@[0.75:0.95]` | Kaggle 제출 기준에 가까운 지표 | 최종 의사결정 지표 |
| Stage별 소요 시간 | 추론/제출 비용 | 병목 확인 |

개선 흐름은 보통 `Stage 2 자동 튜닝 → 상위 trial E2E 재검증 → best 후보 재학습/제출` 순서로 진행합니다.

## Appendix. Stage 2 자동 튜닝 / HPO

이 섹션은 필수 실행 단계가 아닙니다. Stage 2 후보 모델을 탐색하거나 후속 실험을 설계할 때 사용합니다. 기본 objective는 `top1_acc`이며, 상위 trial만 E2E mAP로 재검증하는 방식을 권장합니다.

### search_space 예시

Grid Search와 Optuna는 같은 key 목록을 사용합니다. 차이는 값 형식입니다. nested YAML을 권장하고 dotted key도 옵션으로 지원합니다.

Grid Search:

```yaml
model:
  name: [resnet50, efficientnet_b2]
train:
  lr0: [0.00003, 0.0001, 0.0003]
  lrf: [0.005, 0.01]
  weight_decay: [0.001, 0.01]
  label_smoothing: [0.0, 0.05, 0.1]
```

Optuna:

```yaml
model:
  name:
    type: categorical
    choices: [resnet50, efficientnet_b2, efficientnetv2_s]
train:
  lr0:
    type: float
    low: 0.00001
    high: 0.0003
    log: true
  weight_decay:
    type: float
    low: 0.0001
    high: 0.01
    log: true
  label_smoothing:
    type: float
    low: 0.0
    high: 0.15
```

In [ ]:
# 필요할 때만 RUN_STAGE2_TUNING=True로 변경하세요.
RUN_STAGE2_TUNING = False
TUNING_METHOD = "optuna"  # "grid" 또는 "optuna"
TUNING_EPOCHS = 30
TUNING_TRIALS = 30
TUNING_METRIC = "top1_acc"

# 탐색 공간 예시 — None이면 스크립트 기본값 사용
# Grid Search용 예시 (TUNING_METHOD="grid" 일 때 적용)
GRID_SEARCH_SPACE = {
    "model": {
        "name": ["resnet50", "efficientnet_b2"],
    },
    "train": {
        "lr0": [3e-5, 1e-4, 3e-4],
        "weight_decay": [1e-3, 1e-2],
        "label_smoothing": [0.0, 0.05, 0.1],
    },
}

# Optuna용 예시 (TUNING_METHOD="optuna" 일 때 적용)
OPTUNA_SEARCH_SPACE = {
    "model": {
        "name": {"type": "categorical", "choices": ["resnet50", "efficientnet_b2", "efficientnetv2_s"]},
    },
    "train": {
        "lr0":             {"type": "float", "low": 1e-5,  "high": 3e-4,  "log": True},
        "lrf":             {"type": "float", "low": 5e-3,  "high": 5e-2,  "log": True},
        "weight_decay":    {"type": "float", "low": 1e-4,  "high": 1e-2,  "log": True},
        "label_smoothing": {"type": "float", "low": 0.0,   "high": 0.15},
    },
}

if TUNING_METHOD == "grid":
    TUNING_OUTPUT = PROJECT_ROOT / "experiments/stage2_grid_search"
    _search_space = GRID_SEARCH_SPACE
    tuning_cmd = [
        sys.executable, "scripts/pipeline/stage2_grid_search.py",
        "--base-config", str(S2_CONFIG),
        "--output", str(TUNING_OUTPUT),
        "--data", str(GT_CROP_ROOT),
        "--epochs", str(TUNING_EPOCHS),
        "--metric", TUNING_METRIC,
        "--max-trials", "12",
    ]
else:
    TUNING_OUTPUT = PROJECT_ROOT / "experiments/stage2_optuna"
    _search_space = OPTUNA_SEARCH_SPACE
    tuning_cmd = [
        sys.executable, "scripts/pipeline/stage2_optuna.py",
        "--base-config", str(S2_CONFIG),
        "--output", str(TUNING_OUTPUT),
        "--data", str(GT_CROP_ROOT),
        "--n-trials", str(TUNING_TRIALS),
        "--epochs", str(TUNING_EPOCHS),
        "--metric", TUNING_METRIC,
    ]

# 탐색 공간을 YAML로 써서 --search-space 로 전달
if _search_space is not None:
    import yaml
    TUNING_OUTPUT.mkdir(parents=True, exist_ok=True)
    _space_yaml = TUNING_OUTPUT / "search_space.yaml"
    with open(_space_yaml, "w", encoding="utf-8") as f:
        yaml.safe_dump(_search_space, f, sort_keys=False, allow_unicode=True)
    tuning_cmd += ["--search-space", str(_space_yaml)]
    print(f"[tuning] search_space → {_space_yaml}")

print(" ".join(map(str, tuning_cmd)))
if RUN_STAGE2_TUNING:
    subprocess.run(tuning_cmd, check=True)
else:
    print("튜닝 스킵: RUN_STAGE2_TUNING=False")


## 12. 참고

- 상세 스크립트 사용법: `scripts/README.md`, `scripts/pipeline/README.md`
- 로컬 실행 튜토리얼: `notebooks/pipeline_tutorial.ipynb`
- Colab 런타임이 재시작되면 Bootstrap 셀부터 다시 실행합니다.
